# Priority Analysis 01: Token ID Extraction & Embedding Neighbourhood Test

**Date:** 2026-03-20
**Depends on:** EXP_009d1 results (`stage1_results.pt`)
**Tests:** H3 (Dissolution Pathway Structure)
**Outputs:** `results/` directory with all data and images

## Purpose

Extract the BPE vocabulary indices for our 5 basin attractor tokens and key waypoint tokens,
then check their **embedding neighbourhoods** in `W_E` to distinguish between:

- **(a) Semantic clustering:** neighbours are thematically related (political philosophy, critical theory)
- **(b) BPE substring adjacency:** neighbours are tokens that share character prefixes

If (a): H3 gains evidence. If (b): H3 is challenged.

---

In [ ]:
# ============================================================
# STEP 0: OUTPUT DIRECTORY SETUP
# ============================================================
import os
from pathlib import Path

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)
(OUTPUT_DIR / "data").mkdir(exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
# ============================================================
# STEP 1: SETUP — Load the organism
# ============================================================
import torch
import numpy as np
import json
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Running on: {device}")
print(f"Vocabulary size: {model.cfg.d_vocab}")
print(f"Embedding dimension: {model.cfg.d_model}")

---
## 2. Token ID Extraction

Get the BPE vocabulary index for each basin attractor and key waypoint token.

In [ ]:
# ============================================================
# STEP 2: TOKEN ID EXTRACTION
# ============================================================

# Basin attractor tokens (terminal states)
BASIN_TOKENS = ["prolet", "Divine", "Anarch", "till", "solidarity"]

# Key waypoint tokens (intermediate dissolution pathway)
WAYPOINT_TOKENS = ["capit", "injustice", "Rousse", "Fem", "Ag", "FT",
                   "ash", "Canad", "Zero"]

ALL_TOKENS = BASIN_TOKENS + WAYPOINT_TOKENS

# Extract token IDs
tokenizer = model.tokenizer

md = "## Token ID Registry\n\n"
md += "| Token | BPE Index | Type | Full Decode Check |\n"
md += "|:---|:---|:---|:---|\n"

token_ids = {}
for token_str in ALL_TOKENS:
    ids_no_space = tokenizer.encode(token_str, add_special_tokens=False)
    ids_with_space = tokenizer.encode(" " + token_str, add_special_tokens=False)
    
    if len(ids_no_space) == 1:
        tid = ids_no_space[0]
    elif len(ids_with_space) == 1:
        tid = ids_with_space[0]
    else:
        tid = ids_no_space[0]
    
    decode_check = tokenizer.decode([tid])
    token_type = "BASIN" if token_str in BASIN_TOKENS else "WAYPOINT"
    token_ids[token_str] = tid
    
    md += f"| `{token_str}` | {tid} | {token_type} | `{repr(decode_check)}` |\n"

md += f"\n*Vocabulary size: {model.cfg.d_vocab} tokens*\n"
display(Markdown(md))

print("\nToken IDs dict:")
for k, v in token_ids.items():
    print(f"  {k:>12s} \u2192 {v}")

# === SAVE ===
with open(OUTPUT_DIR / "data" / "token_ids.json", "w") as f:
    json.dump(token_ids, f, indent=2)
print(f"\n\u2705 Saved token IDs to {OUTPUT_DIR / 'data' / 'token_ids.json'}")

---
## 3. Embedding Vectors

Extract the raw embedding vectors from `W_E` and compute basic properties.

In [ ]:
# ============================================================
# STEP 3: EMBEDDING PROPERTIES
# ============================================================

W_E = model.W_E.detach().cpu()  # [vocab_size, 768]

all_norms = W_E.norm(dim=1)
mean_norm = all_norms.mean().item()
std_norm = all_norms.std().item()

norm_ranks = all_norms.argsort(descending=True)
rank_lookup = {tid.item(): rank for rank, tid in enumerate(norm_ranks)}

md = "## Embedding Properties\n\n"
md += f"**Vocabulary stats:** mean norm = {mean_norm:.4f}, std = {std_norm:.4f}\n\n"
md += "| Token | BPE ID | Embedding Norm | Z-score | Norm Rank | Outlier? |\n"
md += "|:---|:---|:---|:---|:---|:---|\n"

embed_props = {}
for token_str in ALL_TOKENS:
    tid = token_ids[token_str]
    norm = all_norms[tid].item()
    z_score = (norm - mean_norm) / std_norm
    rank = rank_lookup[tid] + 1
    percentile = (rank / model.cfg.d_vocab) * 100
    outlier = "\u26a0 YES" if abs(z_score) > 2.0 else ""
    md += f"| `{token_str}` | {tid} | {norm:.4f} | {z_score:+.2f} | {rank} ({percentile:.1f}%) | {outlier} |\n"
    embed_props[token_str] = {
        "id": tid, "norm": round(norm, 4), "z_score": round(z_score, 2),
        "rank": rank, "percentile": round(percentile, 1)
    }

display(Markdown(md))

# === SAVE ===
with open(OUTPUT_DIR / "data" / "embedding_properties.json", "w") as f:
    json.dump({"mean_norm": mean_norm, "std_norm": std_norm, "tokens": embed_props}, f, indent=2)
print(f"\u2705 Saved embedding properties to {OUTPUT_DIR / 'data' / 'embedding_properties.json'}")

---
## 4. The H3 Test: Embedding Neighbourhood Analysis

For each basin/waypoint token, find the **20 nearest neighbours** in the full `W_E` space.

### What we're looking for:
- **Semantic neighbours** (e.g., `prolet` \u2192 `bourgeoisie`) \u2192 supports H3
- **BPE substring neighbours** (e.g., `prolet` \u2192 `proced`) \u2192 challenges H3
- **Random/anomalous neighbours** \u2192 suggests glitch token properties

In [ ]:
# ============================================================
# STEP 4: NEAREST NEIGHBOURS IN W_E
# ============================================================

def get_nearest_neighbours(model, token_id, W_E, k=20):
    """Find k nearest neighbours by cosine similarity in W_E."""
    target_vec = W_E[token_id].unsqueeze(0)
    norms = W_E.norm(dim=1, keepdim=True).clamp(min=1e-8)
    W_E_normed = W_E / norms
    target_normed = target_vec / target_vec.norm().clamp(min=1e-8)
    sims = (W_E_normed @ target_normed.T).squeeze()
    top_sims, top_ids = torch.topk(sims, k + 1)
    results = []
    for sim, idx in zip(top_sims[1:], top_ids[1:]):
        decoded = model.tokenizer.decode([idx.item()])
        results.append({"id": idx.item(), "token": decoded, "cosine_sim": round(sim.item(), 4)})
    return results


all_neighbours = {}
for token_str in ALL_TOKENS:
    tid = token_ids[token_str]
    token_type = "BASIN" if token_str in BASIN_TOKENS else "WAYPOINT"
    neighbours = get_nearest_neighbours(model, tid, W_E, k=20)
    all_neighbours[token_str] = neighbours
    
    md = f"### `{token_str}` (ID: {tid}, {token_type})\n\n"
    md += "| Rank | Neighbour | ID | Cosine Sim | BPE Prefix? |\n"
    md += "|:---|:---|:---|:---|:---|\n"
    
    for i, n in enumerate(neighbours):
        clean_tok = n['token'].replace('\n', '\u21b5').replace('|', '\u2223')
        prefix_match = "\u2713" if (len(token_str) >= 3 and 
                               len(clean_tok.strip()) >= 3 and
                               clean_tok.strip()[:3].lower() == token_str[:3].lower()) else ""
        md += f"| {i+1} | `{clean_tok}` | {n['id']} | {n['cosine_sim']:.4f} | {prefix_match} |\n"
    
    md += "\n"
    display(Markdown(md))

# === SAVE ===
with open(OUTPUT_DIR / "data" / "neighbours.json", "w", encoding="utf-8") as f:
    json.dump(all_neighbours, f, indent=2, ensure_ascii=False)
print(f"\u2705 Saved all neighbour data to {OUTPUT_DIR / 'data' / 'neighbours.json'}")

---
## 5. Cross-Similarity: Basin \u00d7 Waypoint Token Matrix

How similar are the basin/waypoint tokens to *each other* in embedding space?

In [ ]:
# ============================================================
# STEP 5: BASIN/WAYPOINT CROSS-SIMILARITY
# ============================================================
import plotly.express as px
import plotly.io as pio

ids_list = [token_ids[t] for t in ALL_TOKENS]
embeddings = W_E[ids_list]
norms = embeddings.norm(dim=1, keepdim=True).clamp(min=1e-8)
embeddings_normed = embeddings / norms
sim_matrix = (embeddings_normed @ embeddings_normed.T).numpy()

labels = [f"{'B' if t in BASIN_TOKENS else 'W'}:{t}" for t in ALL_TOKENS]

fig = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Basin & Waypoint Token Similarity in W_E (Raw Embeddings)",
    text_auto=".2f",
    aspect="auto",
)
fig.update_layout(template="plotly_dark", height=700, width=700)
fig.show()

# === SAVE ===
pio.write_image(fig, str(OUTPUT_DIR / "images" / "cross_similarity_matrix.png"), scale=2)
fig.write_html(str(OUTPUT_DIR / "images" / "cross_similarity_matrix.html"))
np.save(str(OUTPUT_DIR / "data" / "cross_similarity_matrix.npy"), sim_matrix)
with open(OUTPUT_DIR / "data" / "cross_similarity_labels.json", "w") as f:
    json.dump(labels, f)
print(f"\u2705 Saved cross-similarity matrix (PNG, HTML, NPY) to {OUTPUT_DIR}")

---
## 6. Known Glitch Token Check

In [ ]:
# ============================================================
# STEP 6: GLITCH TOKEN DIAGNOSTICS
# ============================================================

KNOWN_GLITCH_TOKENS = [
    " SolidGoldMagworthy", " petertodd", " StreamerBot",
    " TheNitromeFan", " davidjl", " guaneletters",
    " RandomReddworthy", " embedreportprint",
    " rawdownloadcloneembedreportprint",
    " SolidGoldMag", " exaboraliverably",
]

glitch_ids = set()
for gt in KNOWN_GLITCH_TOKENS:
    ids = tokenizer.encode(gt, add_special_tokens=False)
    glitch_ids.update(ids)

glitch_results = {}
md = "## Glitch Token Check\n\n"
for token_str in ALL_TOKENS:
    tid = token_ids[token_str]
    is_glitch = tid in glitch_ids
    status = "\u26a0 YES" if is_glitch else "\u2713 No"
    md += f"- `{token_str}` (ID {tid}): {status}\n"
    glitch_results[token_str] = {"id": tid, "is_glitch": is_glitch}

md += "\n### Additional Diagnostics\n\n"
md += "| Token | ID | Norm | vs Mean | Norm Rank (of 50,257) |\n"
md += "|:---|:---|:---|:---|:---|\n"

for token_str in ALL_TOKENS:
    tid = token_ids[token_str]
    norm = all_norms[tid].item()
    rank = rank_lookup[tid] + 1
    percentile = (rank / model.cfg.d_vocab) * 100
    md += f"| `{token_str}` | {tid} | {norm:.4f} | {norm/mean_norm:.2f}x | {rank} ({percentile:.1f}%) |\n"

display(Markdown(md))

# === SAVE ===
with open(OUTPUT_DIR / "data" / "glitch_check.json", "w") as f:
    json.dump(glitch_results, f, indent=2)
print(f"\u2705 Saved glitch check to {OUTPUT_DIR / 'data' / 'glitch_check.json'}")

---
## 7. Verdict

### H3 Assessment:

In [ ]:
# ============================================================
# STEP 7: VERDICT
# ============================================================

verdict = {
    "hypothesis": "H3: Intermediate tokens reflect the statistical topology of the training corpus",
    "test": "Embedding neighbourhood analysis in W_E",
    "date": "2026-03-20",
    "result": "SUPPORTED",
    "evidence": [
        {
            "criterion": "Basin token neighbours are semantically related",
            "status": "CONFIRMED (4/5 basins)",
            "detail": "prolet (political philosophy), Divine (theology 20/20), Anarch (political philosophy), solidarity (collective action 20/20, 0 BPE). till is functional/temporal."
        },
        {
            "criterion": "Neighbours are NOT BPE prefix matches only",
            "status": "CONFIRMED",
            "detail": "solidarity has 0/20 BPE prefix matches. prolet top-5 includes bourgeoisie, capitalists (0 shared prefix chars). Divine 18/20 non-prefix."
        },
        {
            "criterion": "Basin tokens are NOT known glitch/anomalous",
            "status": "CONFIRMED",
            "detail": "All 14 tokens passed glitch check. All embedding norms within 1.5 sigma of mean."
        },
        {
            "criterion": "Cross-similarity shows thematic clustering",
            "status": "CONFIRMED",
            "detail": "Political basins cluster (prolet-Anarch 0.47, prolet-solidarity 0.45). Divine isolated (max 0.33). till most isolated. Pathway tokens show structural-to-semantic gradient."
        }
    ],
    "corrections": [
        "capit clusters as CAPITULATION (surrender, succumb, acquiesce), NOT capitalism. Pathway narrative revised."
    ],
    "phase_transition": {
        "structural_phase": ["ash", "Canad", "Ag", "FT", "Zero"],
        "transition_token": "capit",
        "semantic_phase": ["Fem", "injustice", "Rousse"],
        "terminal_basins": ["prolet", "Divine", "Anarch", "solidarity", "till"]
    }
}

md = "## Verdict: H3 — **SUPPORTED**\n\n"
md += "| Evidence | Expected If True | Status |\n"
md += "|:---|:---|:---|\n"
for e in verdict['evidence']:
    md += f"| {e['criterion']} | H3 supported | \u2705 {e['status']} |\n"
md += "\n"
md += "### Key Correction\n"
md += f"\u26a0 {verdict['corrections'][0]}\n\n"
md += "### Dissolution Pathway Phase Transition\n\n"
md += "| Phase | Tokens |\n|:---|:---|\n"
md += f"| Structural (early) | {', '.join(verdict['phase_transition']['structural_phase'])} |\n"
md += f"| **Transition** | **{verdict['phase_transition']['transition_token']}** |\n"
md += f"| Semantic (late) | {', '.join(verdict['phase_transition']['semantic_phase'])} |\n"
md += f"| Terminal basins | {', '.join(verdict['phase_transition']['terminal_basins'])} |\n"

display(Markdown(md))

# === SAVE ===
with open(OUTPUT_DIR / "data" / "verdict.json", "w") as f:
    json.dump(verdict, f, indent=2)
print(f"\u2705 Saved verdict to {OUTPUT_DIR / 'data' / 'verdict.json'}")

# === SAVE SUMMARY MARKDOWN ===
with open(OUTPUT_DIR / "SUMMARY.md", "w", encoding="utf-8") as f:
    f.write("# Priority Analysis 01: Results Summary\n\n")
    f.write(f"**Date:** 2026-03-20\n")
    f.write(f"**Verdict:** H3 SUPPORTED (4/5 basins show strong semantic clustering)\n\n")
    f.write("## Saved Files\n\n")
    f.write("| File | Contents |\n|:---|:---|\n")
    f.write("| `data/token_ids.json` | BPE indices for all 14 tokens |\n")
    f.write("| `data/embedding_properties.json` | Norms, z-scores, ranks |\n")
    f.write("| `data/neighbours.json` | 20 nearest neighbours per token |\n")
    f.write("| `data/cross_similarity_matrix.npy` | 14x14 cosine similarity matrix |\n")
    f.write("| `data/glitch_check.json` | Glitch token check results |\n")
    f.write("| `data/verdict.json` | Full verdict with evidence |\n")
    f.write("| `images/cross_similarity_matrix.png` | Heatmap (2x resolution) |\n")
    f.write("| `images/cross_similarity_matrix.html` | Interactive heatmap |\n")

print(f"\u2705 Saved summary to {OUTPUT_DIR / 'SUMMARY.md'}")
print(f"\n\U0001f3c1 Priority Analysis 01 complete. All outputs in: {OUTPUT_DIR.resolve()}")